# eBay Deal Finder (Python Notebook)

Notebook-first implementation for finding underpriced eBay listings via active-price distribution, sold-volume enrichment, and optional Azure OpenAI listing evaluation.


## 1) Imports and Configuration


In [1]:
import os
import json
import base64
import math
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

import requests
from dotenv import load_dotenv
from openai import (
    AzureOpenAI,
    APIConnectionError,
    APIStatusError,
    APITimeoutError,
    RateLimitError,
)

load_dotenv(override=True)

def require_env(name):
    value = os.environ.get(name, "").strip()
    if not value:
        raise ValueError(f"{name} must be set in .env")
    return value

EBAY_APP_ID = require_env("EBAY_APP_ID")
EBAY_CERT_ID = require_env("EBAY_CERT_ID")
EBAY_SANDBOX = os.environ.get("EBAY_SANDBOX", "false").lower() == "true"
EBAY_MARKETPLACE_ID = os.environ.get("EBAY_MARKETPLACE_ID", "EBAY_US")

AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "").strip()
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY", "").strip()
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "").strip()
LLM_ENABLED = bool(AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY and AZURE_OPENAI_DEPLOYMENT)

SEARCH_QUERY = '"TI-84 Plus CE" graphing calculator'
SEARCH_EXCLUSIONS = '-charger -case -cable -cover -emulator -software -cord -bag -pouch -dock -screen -protector -"parts only" -"for parts" -stand -holder -skin -sticker -keypad'
CONDITION = "Used"
MIN_Z_SCORE = -1.0
EBAY_FEE_RATE = 0.13
RESULTS_PER_PAGE = 200
MAX_LLM_CALLS = 200  # Keep all statistical candidates unless budget is explicitly lower
MAX_IMAGES_PER_LISTING = 5
MIN_MARKET_SAMPLE = 20
MIN_STD_DEV = 0.01
FALLBACK_DISCOUNT_RATE = 0.10
API_TIMEOUT_S = 15
LLM_TIMEOUT_S = 45
API_MAX_RETRIES = 2
API_RETRY_BASE_DELAY_S = 0.5
LLM_CONCURRENCY = 5

EBAY_API_BASE = "https://api.sandbox.ebay.com" if EBAY_SANDBOX else "https://api.ebay.com"
EBAY_AUTH_URL = f"{EBAY_API_BASE}/identity/v1/oauth2/token"
EBAY_BROWSE_URL = f"{EBAY_API_BASE}/buy/browse/v1"

print(f"Environment:  {'SANDBOX' if EBAY_SANDBOX else 'PRODUCTION'}")
print(f"Marketplace:  {EBAY_MARKETPLACE_ID}")
print(f"LLM enabled:  {LLM_ENABLED}")
print(f"Search query: {SEARCH_QUERY}")


/Users/albertjoseph/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Environment:  PRODUCTION
Marketplace:  EBAY_US
LLM enabled:  True
Search query: "TI-84 Plus CE" graphing calculator


## 2) Shared Helpers (Stats, Retry, Ranking)


In [2]:
RETRYABLE_EXCEPTIONS = (
    requests.exceptions.RequestException,
    APITimeoutError,
    APIConnectionError,
    RateLimitError,
    APIStatusError,
)

def format_currency(n):
    return f"${n:.2f}"

def average(numbers):
    return sum(numbers) / len(numbers)

def median(numbers):
    sorted_numbers = sorted(numbers)
    mid = len(sorted_numbers) // 2
    if len(sorted_numbers) % 2 == 0:
        return (sorted_numbers[mid - 1] + sorted_numbers[mid]) / 2
    return sorted_numbers[mid]

def std_dev(numbers, mean):
    squared_diffs = [(n - mean) ** 2 for n in numbers]
    return math.sqrt(sum(squared_diffs) / len(numbers))

def filter_outliers(prices):
    sorted_prices = sorted(prices)
    q1 = sorted_prices[int(len(sorted_prices) * 0.25)]
    q3 = sorted_prices[int(len(sorted_prices) * 0.75)]
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    filtered = [p for p in sorted_prices if lower <= p <= upper]
    return {
        "filtered": filtered,
        "lower": max(lower, 0),
        "upper": upper,
        "removed_count": len(prices) - len(filtered),
    }

def normalize_condition(condition):
    return re.sub(r"\s+", "_", condition.strip().upper())

def extract_status_code(exc):
    if isinstance(exc, requests.exceptions.HTTPError) and exc.response is not None:
        return exc.response.status_code

    for attr in ("status_code", "status", "statusCode"):
        status = getattr(exc, attr, None)
        if isinstance(status, int):
            return status

    response = getattr(exc, "response", None)
    if response is not None:
        status = getattr(response, "status_code", None)
        if isinstance(status, int):
            return status

    return None

def is_retryable_error(exc):
    if isinstance(
        exc,
        (
            requests.exceptions.Timeout,
            requests.exceptions.ConnectionError,
            APITimeoutError,
            APIConnectionError,
            RateLimitError,
        ),
    ):
        return True

    status = extract_status_code(exc)
    if isinstance(status, int):
        return status in (408, 429) or status >= 500

    msg = str(exc).lower()
    return any(token in msg for token in ("timeout", "timed out", "network", "connection reset", "socket"))

def call_with_retry(label, fn, timeout_s, max_retries=API_MAX_RETRIES, base_delay_s=API_RETRY_BASE_DELAY_S):
    for attempt in range(1, max_retries + 2):
        try:
            return fn(timeout_s)
        except RETRYABLE_EXCEPTIONS as exc:
            should_retry = attempt <= max_retries and is_retryable_error(exc)
            if not should_retry:
                raise

            wait_s = base_delay_s * (2 ** (attempt - 1))
            print(
                f"⚠ {label} failed (attempt {attempt}/{max_retries + 1}): {exc}. "
                f"Retrying in {wait_s:.1f}s..."
            )
            time.sleep(wait_s)

def price_rank_from_signal(signal):
    if "Strong" in signal:
        return 3
    if "Buy" in signal:
        return 2
    return 1

def price_only_recommendation(price_rank):
    if price_rank >= 3:
        return "✅ BUY"
    if price_rank >= 2:
        return "⚠️  CONSIDER"
    return "❌ SKIP"

print("✅ Shared helpers loaded")


✅ Shared helpers loaded


## 3) eBay API Client and Shared Item Cache


In [3]:
_EBAY_ACCESS_TOKEN = None
_EBAY_TOKEN_EXPIRES_AT = 0
ITEM_DETAILS_CACHE = {}

def get_ebay_token(timeout_s):
    credentials = base64.b64encode(f"{EBAY_APP_ID}:{EBAY_CERT_ID}".encode()).decode()

    def _request(request_timeout_s):
        response = requests.post(
            EBAY_AUTH_URL,
            headers={
                "Content-Type": "application/x-www-form-urlencoded",
                "Authorization": f"Basic {credentials}",
            },
            data={
                "grant_type": "client_credentials",
                "scope": "https://api.ebay.com/oauth/api_scope",
            },
            timeout=request_timeout_s,
        )
        response.raise_for_status()
        return response.json()

    token_payload = call_with_retry("eBay OAuth token", _request, timeout_s)
    return token_payload["access_token"], int(token_payload.get("expires_in", 0))

def get_ebay_headers(timeout_s=API_TIMEOUT_S):
    global _EBAY_ACCESS_TOKEN
    global _EBAY_TOKEN_EXPIRES_AT

    now = time.time()
    if _EBAY_ACCESS_TOKEN and now < _EBAY_TOKEN_EXPIRES_AT - 60:
        return {
            "Authorization": f"Bearer {_EBAY_ACCESS_TOKEN}",
            "X-EBAY-C-MARKETPLACE-ID": EBAY_MARKETPLACE_ID,
            "Content-Type": "application/json",
        }

    token, expires_in = get_ebay_token(timeout_s)
    _EBAY_ACCESS_TOKEN = token
    _EBAY_TOKEN_EXPIRES_AT = now + expires_in

    return {
        "Authorization": f"Bearer {_EBAY_ACCESS_TOKEN}",
        "X-EBAY-C-MARKETPLACE-ID": EBAY_MARKETPLACE_ID,
        "Content-Type": "application/json",
    }

def browse_search(params, label):
    def _request(request_timeout_s):
        response = requests.get(
            f"{EBAY_BROWSE_URL}/item_summary/search",
            headers=get_ebay_headers(request_timeout_s),
            params=params,
            timeout=request_timeout_s,
        )
        response.raise_for_status()
        return response.json()

    return call_with_retry(label, _request, API_TIMEOUT_S)

def browse_get_item(item_id, label):
    def _request(request_timeout_s):
        response = requests.get(
            f"{EBAY_BROWSE_URL}/item/{item_id}",
            headers=get_ebay_headers(request_timeout_s),
            timeout=request_timeout_s,
        )
        response.raise_for_status()
        return response.json()

    return call_with_retry(label, _request, API_TIMEOUT_S)

def clear_item_details_cache():
    ITEM_DETAILS_CACHE.clear()

def get_item_details(item_id):
    if item_id in ITEM_DETAILS_CACHE:
        return ITEM_DETAILS_CACHE[item_id]

    item_details = browse_get_item(item_id, f"Browse getItem for {item_id}")
    ITEM_DETAILS_CACHE[item_id] = item_details
    return item_details

print("✅ API helpers and item cache loaded")


✅ API helpers and item cache loaded


## 4) Market Sampling and Pricing Analysis


In [4]:
def get_market_listings(query, condition):
    full_query = f"{query} {SEARCH_EXCLUSIONS}" if SEARCH_EXCLUSIONS else query
    condition_filter = normalize_condition(condition)
    all_items = []
    max_items = 10_000
    page = 0

    print(f"\n📊 Fetching active market listings for '{query}' ({condition})...\n")

    while len(all_items) < max_items:
        params = {
            "q": full_query,
            "filter": f"buyingOptions:{{FIXED_PRICE}},conditions:{{{condition_filter}}},priceCurrency:USD",
            "limit": str(RESULTS_PER_PAGE),
            "offset": str(page * RESULTS_PER_PAGE),
        }

        try:
            result = browse_search(params, f"Browse market listings page {page + 1}")
        except RETRYABLE_EXCEPTIONS as exc:
            if not all_items:
                raise RuntimeError(f"Unable to fetch market listings page {page + 1}: {exc}") from exc
            print(
                f"⚠ Market listing fetch stopped at page {page + 1}: {exc}. "
                f"Continuing with {len(all_items)} result(s)."
            )
            break

        items = result.get("itemSummaries") or []
        if not items:
            break

        all_items.extend(items)
        total = result.get("total", 0)
        if len(all_items) >= total:
            break

        page += 1

    return all_items

def analyze_prices(market_items):
    prices = []
    for item in market_items:
        raw_price = item.get("price", {}).get("value")
        try:
            price_value = float(raw_price)
        except (TypeError, ValueError):
            continue
        if price_value > 0:
            prices.append(price_value)

    if not prices:
        print("❌ No market price data found.")
        return None
    if len(prices) < MIN_MARKET_SAMPLE:
        print(
            f"❌ Not enough market listings for reliable pricing "
            f"({len(prices)} found, need at least {MIN_MARKET_SAMPLE})."
        )
        return None

    sorted_prices = sorted(prices)
    print("─" * 50)
    print("  RAW MARKET PRICES (active listings)")
    print("─" * 50)
    print(f"  Items analyzed:  {len(prices)}")
    print(f"  Average price:   {format_currency(average(prices))}")
    print(f"  Median price:    {format_currency(median(prices))}")
    print(f"  Low / High:      {format_currency(sorted_prices[0])} — {format_currency(sorted_prices[-1])}")
    print("─" * 50)

    outlier_result = filter_outliers(prices)
    filtered = outlier_result["filtered"]
    if not filtered:
        print("❌ All prices were filtered as outliers. Check your data.")
        return None
    if len(filtered) < MIN_MARKET_SAMPLE:
        print(
            f"❌ Too few listings after outlier filtering "
            f"({len(filtered)} kept, need at least {MIN_MARKET_SAMPLE})."
        )
        return None

    filtered_sorted = sorted(filtered)
    filtered_mean = average(filtered)
    filtered_median = median(filtered)
    filtered_std_raw = std_dev(filtered, filtered_mean)
    has_usable_std_dev = math.isfinite(filtered_std_raw) and filtered_std_raw >= MIN_STD_DEV
    filtered_std = filtered_std_raw if has_usable_std_dev else 0
    cv = (filtered_std / filtered_mean) if has_usable_std_dev else 0

    stats = {
        "count": len(filtered),
        "average": filtered_mean,
        "median": filtered_median,
        "std_dev": filtered_std,
        "has_usable_std_dev": has_usable_std_dev,
        "cv": cv,
        "low": filtered_sorted[0],
        "high": filtered_sorted[-1],
        "p25": filtered_sorted[int(len(filtered_sorted) * 0.25)],
        "p75": filtered_sorted[int(len(filtered_sorted) * 0.75)],
    }

    print(f"\n  🧹 IQR FILTER: keeping {format_currency(outlier_result['lower'])} — {format_currency(outlier_result['upper'])}")
    print(f"     Removed {outlier_result['removed_count']} outlier(s)\n")
    print("─" * 50)
    print("  FILTERED MARKET VALUE")
    print("─" * 50)
    print(f"  Items kept:      {stats['count']} of {len(prices)}")
    print(f"  Average price:   {format_currency(stats['average'])}")
    print(f"  Median price:    {format_currency(stats['median'])}")
    print(f"  Std deviation:   {format_currency(stats['std_dev'])}")
    print(f"  Low / High:      {format_currency(stats['low'])} — {format_currency(stats['high'])}")
    print(f"  25th / 75th pct: {format_currency(stats['p25'])} — {format_currency(stats['p75'])}")
    print("─" * 50)

    print("\n  📐 PRICE BANDS")
    print("─" * 50)
    if stats["has_usable_std_dev"]:
        print(f"  -2σ (strong buy): {format_currency(stats['median'] - 2 * stats['std_dev'])}")
        print(f"  -1σ (buy):        {format_currency(stats['median'] - 1 * stats['std_dev'])}")
        print(f"   0  (fair value): {format_currency(stats['median'])}")
        print(f"  +1σ (overpriced): {format_currency(stats['median'] + 1 * stats['std_dev'])}")
        print(f"  +2σ (avoid):      {format_currency(stats['median'] + 2 * stats['std_dev'])}")
    else:
        fallback_price = stats['median'] * (1 - FALLBACK_DISCOUNT_RATE)
        print(
            f"  ⚠ Std deviation is below {MIN_STD_DEV}; using discount scoring instead of z-score."
        )
        print(
            f"  Discount threshold ({(FALLBACK_DISCOUNT_RATE * 100):.0f}% below median): {format_currency(fallback_price)}"
        )
    print("─" * 50)

    if not stats["has_usable_std_dev"]:
        market_verdict = "⚠️  Very low variance — using discount-based scoring"
    elif stats["cv"] < 0.3:
        market_verdict = "✅ Tight market — good for arbitrage"
    elif stats["cv"] < 0.5:
        market_verdict = "⚠️  Moderate spread — proceed with caution"
    else:
        market_verdict = "❌ High variance — no reliable fair value"

    cv_display = f"{stats['cv']:.3f}" if stats["has_usable_std_dev"] else "N/A"
    print(f"\n  CV: {cv_display}  →  {market_verdict}")
    print("─" * 50)

    return stats

print("✅ Market analysis functions loaded")


✅ Market analysis functions loaded


## 5) Deal Discovery and Sold-Quantity Enrichment


In [5]:
def find_deal_listings(query, condition, market_stats):
    has_usable_std_dev = market_stats["has_usable_std_dev"]
    raw_max_price = (
        market_stats["median"] + MIN_Z_SCORE * market_stats["std_dev"]
        if has_usable_std_dev
        else market_stats["median"] * (1 - FALLBACK_DISCOUNT_RATE)
    )
    max_price = max(1, math.floor(raw_max_price))
    full_query = f"{query} {SEARCH_EXCLUSIONS}" if SEARCH_EXCLUSIONS else query
    condition_filter = normalize_condition(condition)
    all_items = []
    page = 0
    max_items = 10_000

    threshold_message = (
        f"(z ≤ {MIN_Z_SCORE}, i.e. {abs(MIN_Z_SCORE)}σ below {format_currency(market_stats['median'])} median)"
        if has_usable_std_dev
        else f"({(FALLBACK_DISCOUNT_RATE * 100):.0f}% below {format_currency(market_stats['median'])} median fallback)"
    )
    print(f"\n🔎 Searching active listings under {format_currency(max_price)} {threshold_message}...\n")

    while len(all_items) < max_items:
        params = {
            "q": full_query,
            "filter": (
                f"buyingOptions:{{FIXED_PRICE}},conditions:{{{condition_filter}}},"
                f"price:[..{max_price}],priceCurrency:USD"
            ),
            "sort": "price",
            "limit": str(RESULTS_PER_PAGE),
            "offset": str(page * RESULTS_PER_PAGE),
        }

        try:
            result = browse_search(params, f"Browse deal search page {page + 1}")
        except RETRYABLE_EXCEPTIONS as exc:
            if not all_items:
                raise RuntimeError(f"Unable to fetch deal search page {page + 1}: {exc}") from exc
            print(
                f"⚠ Deal search stopped at page {page + 1}: {exc}. "
                f"Continuing with {len(all_items)} result(s)."
            )
            break

        items = result.get("itemSummaries") or []
        if not items:
            break

        all_items.extend(items)
        total = result.get("total", 0)
        if len(all_items) >= total:
            break

        page += 1

    if not all_items:
        print("😕 No deals found right now. Try again later!\n")
        return []

    deals = []
    for item in all_items:
        try:
            price = float(item.get("price", {}).get("value"))
        except (TypeError, ValueError):
            continue
        if price <= 0:
            continue

        discount_pct = ((market_stats["median"] - price) / market_stats["median"]) * 100
        z_score = ((price - market_stats["median"]) / market_stats["std_dev"]) if has_usable_std_dev else None
        price_score = z_score if has_usable_std_dev else -discount_pct
        gross_profit = market_stats["median"] - price
        fees = market_stats["median"] * EBAY_FEE_RATE
        net_profit = gross_profit - fees

        if has_usable_std_dev:
            if z_score <= -2:
                signal = "🔥 Strong buy"
            elif z_score <= -1:
                signal = "✅ Buy"
            else:
                signal = "⚠️  Marginal"
        else:
            if discount_pct >= 20:
                signal = "🔥 Strong buy"
            elif discount_pct >= 10:
                signal = "✅ Buy"
            else:
                signal = "⚠️  Marginal"

        deals.append(
            {
                "item_id": item.get("itemId", ""),
                "title": item.get("title", ""),
                "price": price,
                "z_score": z_score,
                "discount_pct": discount_pct,
                "price_score": price_score,
                "signal": signal,
                "gross_profit": gross_profit,
                "net_profit": net_profit,
                "link": item.get("itemWebUrl", ""),
            }
        )

    deals.sort(key=lambda d: d["price_score"])
    print(f"📦 Found {len(deals)} deal(s) across {page + 1} page(s)")
    return deals

def enrich_deals(deals):
    print(f"\n📦 Enriching {len(deals)} deals with sold quantity data...\n")
    for deal in deals:
        item_id = deal.get("item_id")
        if not item_id:
            deal["sold_quantity"] = None
            continue

        try:
            details = get_item_details(item_id)
            availabilities = details.get("estimatedAvailabilities") or []
            sold_qty = availabilities[0].get("estimatedSoldQuantity", 0) if availabilities else 0
            deal["sold_quantity"] = sold_qty if isinstance(sold_qty, int) else 0
        except RETRYABLE_EXCEPTIONS as exc:
            deal["sold_quantity"] = None
            print(f"⚠ Sold quantity enrichment failed for {item_id}: {exc}")

    for deal in deals:
        sold = deal["sold_quantity"] if isinstance(deal.get("sold_quantity"), int) else 0
        deal["composite_score"] = deal["price_score"] - 0.1 * sold

    deals.sort(key=lambda d: d["composite_score"])
    print("✅ Enrichment complete")
    return deals

print("✅ Deal discovery and enrichment functions loaded")


✅ Deal discovery and enrichment functions loaded


## 6) Optional Azure OpenAI Listing Evaluation


In [6]:
VALID_VERDICTS = {"BUY", "RISKY", "PASS"}

LLM_STATE_DISABLED = "DISABLED"
LLM_STATE_SKIPPED_BUDGET = "SKIPPED_BUDGET"
LLM_STATE_EVALUATED = "EVALUATED"
LLM_STATE_ERROR = "ERROR"
LLM_STATE_UNSET = "UNSET"

def to_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]

def parse_llm_response(content):
    if not content or not content.strip():
        raise ValueError("LLM returned an empty response.")

    parsed = json.loads(content)
    verdict = str(parsed.get("verdict", "")).upper()
    confidence = float(parsed.get("confidence", -1))
    issues = parsed.get("issues", [])
    summary = str(parsed.get("summary", "")).strip()

    if verdict not in VALID_VERDICTS:
        raise ValueError(f"Invalid verdict: {parsed.get('verdict')}")
    if not (0 <= confidence <= 100):
        raise ValueError(f"Invalid confidence: {parsed.get('confidence')}")
    if not isinstance(issues, list) or any(not isinstance(issue, str) for issue in issues):
        raise ValueError("Invalid issues payload.")
    if not summary:
        raise ValueError("Invalid summary payload.")

    return {
        "verdict": verdict,
        "confidence": round(confidence),
        "issues": [issue.strip() for issue in issues if issue.strip()],
        "summary": summary,
    }

def get_listing_details(item_id):
    item = get_item_details(item_id)
    image_urls = [item.get("image", {}).get("imageUrl")]
    image_urls.extend(img.get("imageUrl") for img in to_list(item.get("additionalImages")) if isinstance(img, dict))

    return {
        "title": item.get("title", ""),
        "description": item.get("description", "") or item.get("shortDescription", ""),
        "image_urls": [url for url in image_urls if url],
        "condition_description": item.get("conditionDescription", ""),
        "condition_name": item.get("condition", ""),
        "item_specifics": [
            {
                "name": aspect.get("name", ""),
                "value": to_list(aspect.get("value")),
            }
            for aspect in to_list(item.get("localizedAspects"))
            if isinstance(aspect, dict)
        ],
        "seller": {
            "user_id": item.get("seller", {}).get("username", ""),
            "feedback_score": item.get("seller", {}).get("feedbackScore", 0),
            "positive_feedback_pct": item.get("seller", {}).get("feedbackPercentage", 0),
        },
    }

def evaluate_listing_with_llm(client, listing_details, deal_context):
    plain_description = re.sub(r"<[^>]*>", " ", listing_details["description"])
    plain_description = re.sub(r"\s+", " ", plain_description).strip()[:3000]

    item_specifics_text = "\n".join(
        f"{spec['name']}: {', '.join(spec['value']) if isinstance(spec['value'], list) else spec['value']}"
        for spec in listing_details["item_specifics"]
    )

    z_context = (
        f"{deal_context['z_score']:.2f} (negative = below market)"
        if deal_context["z_score"] is not None
        else "N/A (market variance too low for z-score)"
    )

    content = [
        {
            "type": "text",
            "text": (
                f"You are an expert eBay arbitrage analyst specializing in evaluating '{SEARCH_QUERY}' listings for resale potential.\n\n"
                f"LISTING DETAILS:\n"
                f"- Title: {listing_details['title']}\n"
                f"- Condition: {listing_details['condition_name']}\n"
                f"- Condition Notes: {listing_details['condition_description'] or 'None provided'}\n"
                f"- Seller: {listing_details['seller']['user_id']} ({listing_details['seller']['positive_feedback_pct']}% positive, {listing_details['seller']['feedback_score']} feedback score)\n"
                f"- Item Specifics:\n{item_specifics_text or 'None listed'}\n\n"
                f"DESCRIPTION:\n{plain_description or 'No description provided'}\n\n"
                f"DEAL CONTEXT:\n"
                f"- Listed price: ${deal_context['price']:.2f}\n"
                f"- Market median: ${deal_context['market_median']:.2f}\n"
                f"- Z-score: {z_context}\n"
                f"- Discount vs median: {deal_context['discount_pct']:.1f}% below median\n"
                f"- Estimated net profit: ${deal_context['net_profit']:.2f}\n\n"
                "Evaluate: product correctness, image condition, red-flag text, seller credibility, accessory completeness, and resale suitability."
            ),
        }
    ]

    for image_url in listing_details["image_urls"][:MAX_IMAGES_PER_LISTING]:
        content.append({"type": "image_url", "image_url": {"url": image_url}})

    def _request(request_timeout_s):
        return client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=[{"role": "user", "content": content}],
            max_completion_tokens=800,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "listing_evaluation",
                    "strict": True,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "verdict": {"type": "string", "enum": ["BUY", "RISKY", "PASS"]},
                            "confidence": {"type": "number", "minimum": 0, "maximum": 100},
                            "issues": {"type": "array", "items": {"type": "string"}},
                            "summary": {"type": "string"},
                        },
                        "required": ["verdict", "confidence", "issues", "summary"],
                        "additionalProperties": False,
                    },
                },
            },
            timeout=request_timeout_s,
        )

    response = call_with_retry(
        f"Azure OpenAI listing evaluation for {listing_details['title'][:30] or 'listing'}",
        _request,
        LLM_TIMEOUT_S,
    )

    return parse_llm_response(response.choices[0].message.content)

def evaluate_deals_with_llm(deals, market_stats):
    for deal in deals:
        deal["llm_state"] = LLM_STATE_UNSET
        deal["llm_verdict"] = "N/A"
        deal["llm_confidence"] = 0
        deal["llm_issues"] = []
        deal["llm_summary"] = "Not evaluated"

    if not LLM_ENABLED:
        for deal in deals:
            deal["llm_state"] = LLM_STATE_DISABLED
            deal["llm_summary"] = "LLM disabled by configuration"
        print(
            "\n💡 LLM evaluation skipped — set AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, "
            "and AZURE_OPENAI_DEPLOYMENT in .env to enable.\n"
        )
        return deals

    client = AzureOpenAI(
        api_version="2024-10-01-preview",
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
    )

    to_evaluate = deals[:MAX_LLM_CALLS]
    skipped = deals[MAX_LLM_CALLS:]
    for deal in skipped:
        deal["llm_state"] = LLM_STATE_SKIPPED_BUDGET
        deal["llm_summary"] = "Skipped due LLM budget cap"

    if skipped:
        print(
            f"  ℹ️  {len(deals)} deals passed the statistical filter; evaluating top {len(to_evaluate)} "
            f"(budget cap: {MAX_LLM_CALLS})."
        )
        print(
            f"     Remaining {len(skipped)} deal(s) will use price-signal-only recommendations.\n"
        )

    print(f"\n🤖 Evaluating {len(to_evaluate)} deals with Azure OpenAI ({LLM_CONCURRENCY} parallel)...\n")

    progress_lock = Lock()
    completed = [0]

    def evaluate_one(deal, index):
        label = f"  [{index + 1}/{len(to_evaluate)}] {deal['title'][:40]}..."
        try:
            listing_details = get_listing_details(deal["item_id"])
            evaluation = evaluate_listing_with_llm(
                client,
                listing_details,
                {
                    "price": deal["price"],
                    "market_median": market_stats["median"],
                    "z_score": deal["z_score"],
                    "discount_pct": deal["discount_pct"],
                    "net_profit": deal["net_profit"],
                },
            )
            deal["llm_state"] = LLM_STATE_EVALUATED
            deal["llm_verdict"] = evaluation["verdict"]
            deal["llm_confidence"] = evaluation["confidence"]
            deal["llm_issues"] = evaluation["issues"]
            deal["llm_summary"] = evaluation["summary"]

            icon = {"BUY": "🟢", "RISKY": "🟡", "PASS": "🔴"}[evaluation["verdict"]]
            with progress_lock:
                completed[0] += 1
                progress = completed[0]
            print(f"{label} {icon} {evaluation['verdict']} ({evaluation['confidence']}%) [{progress}/{len(to_evaluate)}]")
        except (
            ValueError,
            KeyError,
            TypeError,
            requests.exceptions.RequestException,
            APITimeoutError,
            APIConnectionError,
            RateLimitError,
            APIStatusError,
        ) as exc:
            deal["llm_state"] = LLM_STATE_ERROR
            deal["llm_verdict"] = "N/A"
            deal["llm_confidence"] = 0
            deal["llm_issues"] = []
            deal["llm_summary"] = f"Error: {exc}"
            with progress_lock:
                completed[0] += 1
                progress = completed[0]
            print(f"{label} ⚪ Error: {str(exc)[:60]} [{progress}/{len(to_evaluate)}]")

    with ThreadPoolExecutor(max_workers=LLM_CONCURRENCY) as executor:
        futures = [executor.submit(evaluate_one, deal, idx) for idx, deal in enumerate(to_evaluate)]
        for future in as_completed(futures):
            future.result()

    return deals

print("✅ LLM evaluation functions loaded")


✅ LLM evaluation functions loaded


## 7) Final Ranking, Recommendations, and Display


In [7]:
def apply_recommendations(deals):
    for deal in deals:
        price_rank = price_rank_from_signal(deal.get("signal", ""))
        llm_state = deal.get("llm_state", LLM_STATE_UNSET)
        llm_verdict = deal.get("llm_verdict", "N/A")
        confidence = deal.get("llm_confidence", 0)

        llm_rank = -1
        if llm_state == LLM_STATE_EVALUATED:
            llm_rank = {"BUY": 3, "RISKY": 2, "PASS": 0}.get(llm_verdict, -1)

        llm_bonus = llm_rank if llm_rank > 0 else 0
        confidence_bonus = (confidence / 100) if llm_state == LLM_STATE_EVALUATED else 0
        deal["final_score"] = price_rank + llm_bonus + confidence_bonus

        if llm_state in (LLM_STATE_DISABLED, LLM_STATE_SKIPPED_BUDGET, LLM_STATE_UNSET):
            deal["final_rec"] = price_only_recommendation(price_rank)
        elif llm_state == LLM_STATE_ERROR:
            deal["final_rec"] = "⚠️  CONSIDER" if price_rank >= 2 else "❌ SKIP"
        elif llm_rank == 0:
            deal["final_rec"] = "❌ SKIP"
        elif llm_rank == 3 and price_rank >= 2:
            deal["final_rec"] = "🏆 STRONG BUY"
        elif llm_rank == 3:
            deal["final_rec"] = "✅ BUY"
        elif llm_rank == 2 and price_rank >= 2:
            deal["final_rec"] = "⚠️  CONSIDER"
        else:
            deal["final_rec"] = "❌ SKIP"

    # Sort by recommendation tier first (STRONG BUY > BUY > CONSIDER > SKIP), then by score
    REC_ORDER = {"🏆 STRONG BUY": 0, "✅ BUY": 1, "⚠️  CONSIDER": 2, "❌ SKIP": 3}
    deals.sort(key=lambda d: (REC_ORDER.get(d["final_rec"], 9), -d["final_score"]))
    return deals

def display_results(deals, market_stats):
    deals = apply_recommendations(deals)
    has_usable = market_stats["has_usable_std_dev"]
    show_llm_columns = any(d.get("llm_state") != LLM_STATE_DISABLED for d in deals)
    score_header = "Z-SCORE" if has_usable else "DISC%"

    print(f"\n📊 Full Analysis — {len(deals)} deals (best first):\n")
    rec_col = "RECOMMENDATION".ljust(18)
    header = (
        rec_col
        + score_header.ljust(10)
        + "SOLD".ljust(7)
        + (("VERDICT".ljust(14)) if show_llm_columns else "")
        + "SIGNAL".ljust(16)
        + "PRICE".ljust(10)
        + "NET PROFIT".ljust(12)
        + "TITLE".ljust(45)
        + "LINK"
    )
    print(header)
    print("─" * 150)

    for deal in deals:
        state = deal.get("llm_state", LLM_STATE_UNSET)
        verdict_col = ""
        if show_llm_columns:
            if state == LLM_STATE_EVALUATED:
                icon = {"BUY": "🟢", "RISKY": "🟡", "PASS": "🔴"}.get(deal.get("llm_verdict"), "⚪")
                verdict_col = f"{icon} {deal.get('llm_verdict', 'N/A')}".ljust(14)
            elif state == LLM_STATE_SKIPPED_BUDGET:
                verdict_col = "⚪ BUDGET".ljust(14)
            elif state == LLM_STATE_ERROR:
                verdict_col = "⚪ ERROR".ljust(14)
            else:
                verdict_col = "⚪ N/A".ljust(14)

        score_value = (
            f"{deal['z_score']:.2f}" if has_usable and deal.get("z_score") is not None else f"{deal['discount_pct']:.1f}%"
        )
        sold_col = str(deal["sold_quantity"]) if isinstance(deal.get("sold_quantity"), int) else "?"

        print(
            deal["final_rec"].ljust(18)
            + score_value.ljust(10)
            + sold_col.ljust(7)
            + verdict_col
            + deal["signal"].ljust(16)
            + format_currency(deal["price"]).ljust(10)
            + f"{('+' if deal['net_profit'] > 0 else '')}{format_currency(deal['net_profit'])}".ljust(12)
            + deal["title"][:42].ljust(45)
            + deal.get("link", "")
        )

    print("─" * 150)
    print("\n💡 Column guide:")
    if has_usable:
        print("   Z-SCORE  = (price - median) / std dev — lower is cheaper")
    else:
        print("   DISC%    = ((median - price) / median) × 100")
    print("   SOLD     = units sold on this listing (demand signal)")
    if show_llm_columns:
        print("   VERDICT  = LLM state/verdict: evaluated, skipped-by-budget, or error")
        print("   RECOMMENDATION = combined price + LLM result when evaluated")
    print(f"   NET PROFIT = (median - price) minus ~{EBAY_FEE_RATE * 100:.0f}% eBay fees (excl. shipping)\n")

    actionable = [d for d in deals if d["final_rec"] in ("🏆 STRONG BUY", "✅ BUY")]
    consider = [d for d in deals if d["final_rec"] == "⚠️  CONSIDER"]
    skipped = sum(1 for d in deals if d["final_rec"] == "❌ SKIP")

    print("═" * 70)
    print("  🎯 ACTION ITEMS — Listings to buy NOW")
    print("═" * 70)
    if not actionable:
        print("\n  No strong recommendations right now.")
    else:
        for i, deal in enumerate(actionable, start=1):
            score_summary = (
                f"z={deal['z_score']:.2f}" if has_usable and deal.get("z_score") is not None else f"{deal['discount_pct']:.1f}% off"
            )
            print(f"\n  {deal['final_rec']}  [{i}]")
            print(f"  📦 {deal['title'][:70]}")
            print(
                f"  💰 {format_currency(deal['price'])} → est. profit "
                f"{('+' if deal['net_profit'] > 0 else '')}{format_currency(deal['net_profit'])} ({score_summary})"
            )
            if deal.get("llm_summary") and not deal["llm_summary"].startswith("Not evaluated"):
                print(f"  🤖 {deal['llm_summary']}")
            print(f"  🔗 {deal.get('link', '')}")

    if consider:
        print("\n" + "─" * 70)
        print("  ⚠️  WORTH CONSIDERING (proceed with caution)")
        print("─" * 70)
        for i, deal in enumerate(consider, start=1):
            score_summary = (
                f"z={deal['z_score']:.2f}" if has_usable and deal.get("z_score") is not None else f"{deal['discount_pct']:.1f}% off"
            )
            print(f"\n  [{i}] {deal['title'][:65]}")
            print(
                f"      {format_currency(deal['price'])} | profit: "
                f"{('+' if deal['net_profit'] > 0 else '')}{format_currency(deal['net_profit'])} ({score_summary})"
            )
            if deal.get("llm_summary") and not deal["llm_summary"].startswith("Not evaluated"):
                print(f"      🤖 {deal['llm_summary']}")
            for issue in deal.get("llm_issues", []):
                print(f"      ⚠ {issue}")
            print(f"      🔗 {deal.get('link', '')}")

    print(f"\n  📊 Summary: {len(actionable)} BUY | {len(consider)} CONSIDER | {skipped} SKIP out of {len(deals)} deals")
    print("═" * 70 + "\n")

print("✅ Recommendation and display functions loaded")


✅ Recommendation and display functions loaded


## 8) Execute Full Pipeline


In [ ]:
print("═" * 50)
print("  eBay Deal Finder — Python Notebook")
print("═" * 50)
print(f"  Query:     '{SEARCH_QUERY}'")
print(f"  Condition: {CONDITION}")
print(f"  Min z:     {MIN_Z_SCORE} ({abs(MIN_Z_SCORE)}σ below median)")
print("═" * 50)

clear_item_details_cache()

market_items = get_market_listings(SEARCH_QUERY, CONDITION)
if not market_items:
    raise RuntimeError("No market listings found. Check your query or API credentials.")

market_stats = analyze_prices(market_items)
if not market_stats:
    raise RuntimeError("Market analysis did not produce usable pricing statistics.")

deals = find_deal_listings(SEARCH_QUERY, CONDITION, market_stats)
if deals:
    deals = enrich_deals(deals)
    deals = evaluate_deals_with_llm(deals, market_stats)
    display_results(deals, market_stats)
else:
    print("No qualifying deals returned for the configured threshold.")


══════════════════════════════════════════════════
  eBay Deal Finder — Python Notebook
══════════════════════════════════════════════════
  Query:     '"TI-84 Plus CE" graphing calculator'
  Condition: Used
  Min z:     -1.0 (1.0σ below median)
══════════════════════════════════════════════════

📊 Fetching active market listings for '"TI-84 Plus CE" graphing calculator' (Used)...

──────────────────────────────────────────────────
  RAW MARKET PRICES (active listings)
──────────────────────────────────────────────────
  Items analyzed:  424
  Average price:   $83.31
  Median price:    $75.00
  Low / High:      $7.00 — $899.00
──────────────────────────────────────────────────

  🧹 IQR FILTER: keeping $27.50 — $127.50
     Removed 29 outlier(s)

──────────────────────────────────────────────────
  FILTERED MARKET VALUE
──────────────────────────────────────────────────
  Items kept:      395 of 424
  Average price:   $76.89
  Median price:    $75.00
  Std deviation:   $18.86
  Low / Hi

## 9) Deterministic Sanity Checks (No Network)


In [ ]:
def run_deterministic_checks():
    outliers = filter_outliers([10, 11, 12, 13, 14, 999])
    assert 999 not in outliers["filtered"], "Outlier filter should remove obvious outliers"

    sample_deals = [
        {
            "id": "disabled-strong",
            "signal": "🔥 Strong buy",
            "llm_state": LLM_STATE_DISABLED,
            "llm_verdict": "N/A",
            "llm_confidence": 0,
            "discount_pct": 25,
        },
        {
            "id": "budget-buy",
            "signal": "✅ Buy",
            "llm_state": LLM_STATE_SKIPPED_BUDGET,
            "llm_verdict": "N/A",
            "llm_confidence": 0,
            "discount_pct": 12,
        },
        {
            "id": "pass-strong",
            "signal": "🔥 Strong buy",
            "llm_state": LLM_STATE_EVALUATED,
            "llm_verdict": "PASS",
            "llm_confidence": 99,
            "discount_pct": 30,
        },
        {
            "id": "error-buy",
            "signal": "✅ Buy",
            "llm_state": LLM_STATE_ERROR,
            "llm_verdict": "N/A",
            "llm_confidence": 0,
            "discount_pct": 11,
        },
        {
            "id": "eval-buy",
            "signal": "✅ Buy",
            "llm_state": LLM_STATE_EVALUATED,
            "llm_verdict": "BUY",
            "llm_confidence": 90,
            "discount_pct": 15,
        },
    ]

    for sample in sample_deals:
        sample["price"] = 50
        sample["z_score"] = -1.2
        sample["sold_quantity"] = 0
        sample["title"] = sample["id"]
        sample["net_profit"] = 10
        sample["link"] = ""
        sample["llm_issues"] = []
        sample["llm_summary"] = ""

    ranked = apply_recommendations(sample_deals)
    by_id = {deal["id"]: deal for deal in ranked}

    assert by_id["disabled-strong"]["final_rec"] == "✅ BUY"
    assert by_id["budget-buy"]["final_rec"] == "⚠️  CONSIDER"
    assert by_id["pass-strong"]["final_rec"] == "❌ SKIP"
    assert by_id["error-buy"]["final_rec"] == "⚠️  CONSIDER"
    assert by_id["eval-buy"]["final_rec"] in ("🏆 STRONG BUY", "✅ BUY")

    print("✅ Deterministic checks passed")

run_deterministic_checks()
